# Notebook 22 — WU-20C pinned nnU-Net runtime qualification

**CPU only, no training, no held-out labels, no model inference.** This validates exact nnU-Net 2.8.1 normalization and official held-out preprocessing on a bounded patient-image region for both prospective Dataset506/507 arms. Full-volume inference, trained model checkpoints, and independent clinical mask review remain blocked.

*Run all cells with a CPU runtime and connected Google Drive.*

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Frozen source verification and pinned code

Only source images, frozen plans and training-fold calibration JSON are read; no reference lesion labels, old model predictions, or patient outcomes are consulted.

In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path
ROOT=Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
RAW=ROOT/'nnUNet_raw'/'Dataset505_ISLES2024_NCCT_PERFUSION'
SPLITS=ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'/'splits_final.json'
PLANS=ROOT/'nnUNet_preprocessed'/'Dataset505_ISLES2024_NCCT_PERFUSION'/'nnUNetPlans.json'
FOLD=2
CASE='sub_stroke0016'
FOLD_JSON=ROOT/'results'/'wu20b_full_fold_calibration'/'fold_private'/f'WU20B_fold{FOLD}_parameters_PRIVATE.json'
OUTPUT=ROOT/'results'/'wu20c_nnunet_runtime_qualification'
SOURCE=Path('/content/wu20c_pinned_source')
PACKAGE=SOURCE/'src'/'hybridstrokeseg'
PACKAGE.mkdir(parents=True,exist_ok=True)
(SOURCE/'scripts').mkdir(parents=True,exist_ok=True)
def sha256(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for b in iter(lambda:f.read(1<<20),b''):h.update(b)
    return h.hexdigest()
assert RAW.is_dir() and SPLITS.is_file() and PLANS.is_file() and FOLD_JSON.is_file()
assert sha256(SPLITS)=='664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
assert sha256(PLANS)=='6203521136de2f57bbe3895a639863112a5869697de9b752ade42dc1a23e4dca'
COMMIT='47b59e34a2cbdd5d342b026e84fed1bad091cd64'
BASE=f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{COMMIT}'
for path in (
    'src/hybridstrokeseg/wu19_normalization_protocol.py',
    'src/hybridstrokeseg/wu20_fold_preprocessing.py',
    'scripts/qualify_isles2024_nnunet_runtime.py',
):
    destination=SOURCE/path
    destination.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['curl','-fLsS','--retry','5','--retry-delay','3',f'{BASE}/{path}','-o',str(destination)],check=True)
    assert destination.stat().st_size>500, f'Source missing/truncated: {path}'
    print('Pinned source',path,sha256(destination)[:16])
print('Frozen split and plans: verified')

## 2. Install the exact nnU-Net release

Requires `nnunetv2==2.8.1`. An incompatible install must fail, not fall back to an unpinned release.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','--disable-pip-version-check',
    'nnunetv2==2.8.1','nibabel>=5.2,<6'],check=True)
import importlib.metadata
assert importlib.metadata.version('nnunetv2')=='2.8.1'
print('Runtime:',importlib.metadata.version('nnunetv2'))

## 3. Run official held-out preprocessor and check pairwise parity

The working ROI is derived only from the held-out acute NCCT image. **It is not a full-volume clinical validation.** The original image is never changed. Memory-intensive transformed NIfTI files remain in temporary `/content/` storage and are removed after the test.

In [ ]:
import tempfile
sys.path.insert(0,str(SOURCE/'src'))
COMMAND=[sys.executable,'-u',str(SOURCE/'scripts'/'qualify_isles2024_nnunet_runtime.py'),
  '--raw505',str(RAW),'--splits',str(SPLITS),'--source-plans',str(PLANS),
  '--fold-manifest',str(FOLD_JSON),'--output',str(OUTPUT),'--fold',str(FOLD),'--case',CASE]
OUTPUT.mkdir(parents=True,exist_ok=True)
LOG=OUTPUT/'WU20C_runtime_qualification.log'
env=os.environ.copy()
env['PYTHONPATH']=str(SOURCE/'src')+os.pathsep+env.get('PYTHONPATH','')
with LOG.open('w',encoding='utf8') as log:
    with subprocess.Popen(COMMAND,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          text=True,bufsize=1) as p:
        for line in p.stdout:
            print(line,end='',flush=True)
            log.write(line);log.flush()
        returncode=p.wait()
if returncode:
    raise RuntimeError(f'WU20C technical qualification failed: exit={returncode}. Check {LOG}')
report=json.loads((OUTPUT/'WU20C_probe_aggregate.json').read_text())
assert report['training_ready'] is False
assert report['real_model_inference_performed'] is False
assert report['locked_architecture_spacing'] is True
assert report['postresample_ncct_max_abs_difference']<=1e-5
print('CPU preprocessing technical probe completed; GPU training still BLOCKED')

## 4. Prepare independent anatomical mask review checklist

**No automated clinical approval.** Preserve reviewer edits between reruns. The checklist refers to the 15 existing WU20B support overlays, which stay private in Google Drive.

In [ ]:
import csv
OVERLAYS=ROOT/'results'/'wu20b_full_fold_calibration'/'private_mask_overlays'
REVIEW=OUTPUT/'WU20C_mask_review_checklist_PRIVATE.csv'
images=sorted(OVERLAYS.glob('*_support_QC.png'))
assert len(images)==15, f'Expected 15 image-only mask overlays, found {len(images)}'
fields=['case_id','overlay_filename','decision','brain_tissue_preserved','extracranial_tissue_included',
        'correction_required','reviewer','review_date','notes']
if REVIEW.exists():
    with REVIEW.open(encoding='utf8',newline='') as f:
        existing=list(csv.DictReader(f))
    assert {r['case_id'] for r in existing}=={p.name.split('_support_QC')[0] for p in images},'Review case inventory changed'
    print('Preserved existing reviewer checklist:', REVIEW)
else:
    with REVIEW.open('w',encoding='utf8',newline='') as f:
        writer=csv.DictWriter(f,fieldnames=fields)
        writer.writeheader()
        for path in images:
            writer.writerow({'case_id':path.name.split('_support_QC')[0],
                             'overlay_filename':path.name,'decision':'PENDING',
                             'brain_tissue_preserved':'UNKNOWN','extracranial_tissue_included':'UNKNOWN',
                             'correction_required':'UNKNOWN','reviewer':'','review_date':'','notes':''})
    print('Created independent anatomical mask review checklist:',REVIEW)
print('Completed technical probe:',OUTPUT/'WU20C_probe_aggregate.json')
print('Private full probe:',OUTPUT/'WU20C_probe_PRIVATE.json')
print('Mask review images:',OVERLAYS)
print('Training approval: STILL BLOCKED')